# 🔍 Automated Metadata Generation System
**Supports:** PDF · DOCX · TXT · PNG/JPG  
**Extracts:** Keywords · Named Entities · Summary · Quality Scores · JSON Export

> **Fixes applied vs original notebook**
> - Removed duplicate `import pandas` cell (cell 0)
> - Replaced `!pip install` with a clean, deduplicated install block
> - Fixed `KeyBERT.extract_keywords` — removed invalid `top_k` arg (use `top_n`)
> - Fixed `TfidfVectorizer` — `min_df=2` crashes on single-doc corpus → changed to `1`
> - Removed Colab-specific hardcoded paths (`/content/…`)
> - Removed broken `ipywidgets` upload cell (not needed for local/Colab use)
> - Added `nltk.download` guard so it never re-downloads on repeated runs
> - Fixed `textwrap` import (was inside function body without top-level import)
> - Removed stale Streamlit draft cell (that belongs in `app.py`)
> - All classes and functions are now in logical, dependency-safe order

## 1 · Install Dependencies

In [ ]:
# Run once – safe to re-run (pip skips already-installed packages)
import sys

packages = [
    "transformers",
    "torch",
    "spacy",
    "keybert",
    "pytesseract",
    "pdfplumber",
    "PyMuPDF",
    "docx2txt",
    "opencv-python",
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "nltk",
    "scikit-learn",
    "sentence-transformers",
]

import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)

# Download spaCy English model
subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
print("✅ All packages installed.")

## 2 · Imports & Global Setup

In [ ]:
# Standard library
import os
import re
import json
import textwrap
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# Data
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use("default")
sns.set_palette("husl")

# Document parsing
import fitz          # PyMuPDF
import pdfplumber
import docx2txt
import pytesseract
from PIL import Image
import cv2

# NLP
import spacy
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from keybert import KeyBERT
from transformers import pipeline

# ── NLTK data (idempotent) ──────────────────────────────────
for pkg in ("punkt", "punkt_tab", "stopwords", "averaged_perceptron_tagger"):
    nltk.download(pkg, quiet=True)

print("✅ All imports successful.")

## 3 · Initialise NLP Models

In [ ]:
# spaCy
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy model loaded")
except OSError:
    print("⚠️  Run:  python -m spacy download en_core_web_sm")
    nlp = None

# KeyBERT
kw_model = KeyBERT()
print("✅ KeyBERT initialised")

# Summariser  (facebook/bart-large-cnn  ~1.6 GB – cached after first download)
try:
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
    print("✅ Summarisation model loaded")
except Exception as e:
    print(f"⚠️  Summariser unavailable: {e}")
    summarizer = None

# Stopwords
try:
    stop_words = set(stopwords.words("english"))
    print("✅ Stopwords loaded")
except Exception as e:
    print(f"⚠️  Stopwords error: {e}")
    stop_words = set()

## 4 · File Loader

In [ ]:
SUPPORTED_EXTENSIONS = {".pdf", ".docx", ".txt", ".png", ".jpg", ".jpeg"}

def load_file_info(file_path: str) -> dict | None:
    """Validate a local file path and return a metadata dict."""
    p = Path(file_path)
    if not p.exists():
        print(f"❌ File not found: {file_path}")
        return None
    if p.suffix.lower() not in SUPPORTED_EXTENSIONS:
        print(f"❌ Unsupported type: {p.suffix}")
        return None
    info = {
        "name":      p.name,
        "path":      str(p),
        "extension": p.suffix.lower(),
        "size":      p.stat().st_size,
    }
    print(f"✅ {p.name}  ({info['size'] / 1024:.1f} KB  |  {p.suffix})")
    return info

## 5 · Text Extraction

In [ ]:
class PDFTextExtractor:
    """Dual-engine PDF extractor (PyMuPDF preferred, pdfplumber as fallback)."""

    def __init__(self):
        self.doc_metadata: dict = {}

    def _pymupdf(self, path: str) -> str:
        doc = fitz.open(path)
        self.doc_metadata = {
            "title":             doc.metadata.get("title", ""),
            "author":            doc.metadata.get("author", ""),
            "subject":           doc.metadata.get("subject", ""),
            "creator":           doc.metadata.get("creator", ""),
            "producer":          doc.metadata.get("producer", ""),
            "creation_date":     doc.metadata.get("creationDate", ""),
            "modification_date": doc.metadata.get("modDate", ""),
            "page_count":        doc.page_count,
        }
        text = "\n".join(doc.load_page(i).get_text() for i in range(doc.page_count))
        doc.close()
        return text

    def _pdfplumber(self, path: str) -> str:
        with pdfplumber.open(path) as pdf:
            return "\n".join(p.extract_text() or "" for p in pdf.pages)

    def extract(self, path: str) -> tuple[str, dict]:
        print(f"🔄 PDF extraction: {Path(path).name}")
        t1, t2 = "", ""
        try:   t1 = self._pymupdf(path)
        except Exception as e: print(f"  PyMuPDF: {e}")
        try:   t2 = self._pdfplumber(path)
        except Exception as e: print(f"  pdfplumber: {e}")

        best, method = (t1, "PyMuPDF") if len(t1) >= len(t2) else (t2, "pdfplumber")
        print(f"  ✅ {method}  –  {len(best):,} chars")
        return best, self.doc_metadata


class DOCXTextExtractor:
    def extract(self, path: str) -> tuple[str, dict]:
        print(f"🔄 DOCX extraction: {Path(path).name}")
        text = docx2txt.process(path) or ""
        meta = {
            "word_count":      len(text.split()),
            "character_count": len(text),
        }
        print(f"  ✅ {len(text):,} chars")
        return text, meta


class OCRTextExtractor:
    def _preprocess(self, img_path: str):
        img    = cv2.imread(img_path)
        gray   = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        blur   = cv2.GaussianBlur(gray, (3, 3), 0)
        _, thr = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        opened = cv2.morphologyEx(thr, cv2.MORPH_OPEN, kernel, iterations=1)
        return 255 - opened

    def extract(self, path: str) -> tuple[str, dict]:
        print(f"🔄 OCR extraction: {Path(path).name}")
        processed = self._preprocess(path)
        text = pytesseract.image_to_string(processed, config="--psm 6", lang="eng")
        with Image.open(path) as im:
            meta = {"image_size": im.size, "image_mode": im.mode}
        print(f"  ✅ {len(text):,} chars")
        return text, meta


# Singleton instances
pdf_extractor  = PDFTextExtractor()
docx_extractor = DOCXTextExtractor()
ocr_extractor  = OCRTextExtractor()


def extract_text_from_file(file_info: dict) -> tuple[str, dict]:
    """Route to the correct extractor based on file extension."""
    if not file_info:
        return "", {}

    ext  = file_info["extension"]
    path = file_info["path"]

    print(f"\n📄 {file_info['name']}  ({ext})")

    if ext == ".pdf":
        text, meta = pdf_extractor.extract(path)
    elif ext == ".docx":
        text, meta = docx_extractor.extract(path)
    elif ext == ".txt":
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        meta = {"word_count": len(text.split()), "character_count": len(text)}
        print(f"  ✅ {len(text):,} chars")
    elif ext in (".png", ".jpg", ".jpeg"):
        text, meta = ocr_extractor.extract(path)
    else:
        print(f"❌ Unsupported: {ext}")
        return "", {}

    meta.update({
        "original_filename": file_info["name"],
        "file_extension":    ext,
    })
    return text, meta

## 6 · Text Preprocessing

In [ ]:
class TextPreprocessor:
    """Clean, normalise, and sentence-tokenise raw extracted text."""

    CUSTOM_STOPWORDS = {
        "page", "pages", "document", "file", "pdf", "docx",
        "figure", "table", "section", "chapter", "appendix",
    }
    CONTRACTIONS = {
        "won't": "will not", "can't": "cannot", "n't": " not",
        "'re": " are", "'ve": " have", "'ll": " will",
        "'d": " would", "'m": " am",
    }

    def __init__(self):
        self.stop_words = set(stopwords.words("english")) | self.CUSTOM_STOPWORDS

    # ── cleaning steps ───────────────────────────────────────
    def _basic_clean(self, text: str) -> str:
        text = re.sub(r"\s+",           " ",   text)
        text = re.sub(r"[^\w\s\.\,\!\?\;\:\-\(\)]", " ", text)
        text = re.sub(r"\s+([.!?])",    r"\1", text)
        text = re.sub(r"([.!?])\s*([A-Z])", r"\1 \2", text)
        text = re.sub(r"[.]{2,}",        ".",   text)
        return text.strip()

    def _remove_noise(self, text: str) -> str:
        text = re.sub(r"\bPage\s+\d+\b",    "",  text, flags=re.IGNORECASE)
        text = re.sub(r"\b\d+\s+of\s+\d+\b", "", text, flags=re.IGNORECASE)
        text = re.sub(r"http\S+",               "",  text)
        text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "", text)
        text = re.sub(r"^\s*\d+\.\s*",       "",  text, flags=re.MULTILINE)
        text = re.sub(r"\t+",                  " ",  text)
        text = re.sub(r" {3,}",                 " ",  text)
        return text

    def _normalise(self, text: str) -> str:
        text = text.lower()
        for c, e in self.CONTRACTIONS.items():
            text = text.replace(c, e)
        text = re.sub(r"\$\d+(?:,\d{3})*(?:\.\d{2})?", "CURRENCY", text)
        text = re.sub(r"\b\d{4}\b",                       "YEAR",     text)
        text = re.sub(r"\b\d+(?:,\d{3})*(?:\.\d+)?\b",  "NUMBER",   text)
        return text

    def _quality_sentences(self, text: str) -> list[str]:
        return [
            s.strip()
            for s in sent_tokenize(text)
            if 10 <= len(s) <= 500
            and len(re.findall(r"[a-zA-Z]", s)) > len(s) * 0.7
        ]

    # ── public API ───────────────────────────────────────────
    def preprocess(self, text: str, normalise: bool = True) -> dict:
        if not text:
            return {"processed_text": "", "sentences": [], "word_count": 0, "character_count": 0}

        print("🔄 Preprocessing …")
        t = self._basic_clean(text)
        t = self._remove_noise(t)
        sentences = self._quality_sentences(t)

        if normalise:
            t = self._normalise(t)

        result = {
            "processed_text":  t,
            "sentences":       sentences,
            "word_count":      len(t.split()),
            "character_count": len(t),
        }
        print(f"  ✅ {result['word_count']:,} words  |  {len(sentences):,} sentences")
        return result


text_preprocessor = TextPreprocessor()

## 7 · Text Quality Assessment

In [ ]:
def assess_text_quality(original: str, processed: str) -> dict:
    metrics: dict = {
        "original_length":  len(original)  if original  else 0,
        "processed_length": len(processed) if processed else 0,
    }
    metrics["compression_ratio"] = (
        metrics["processed_length"] / metrics["original_length"]
        if metrics["original_length"] > 0 else 0
    )
    if processed:
        words = processed.split()
        sentences = sent_tokenize(processed)
        metrics.update({
            "word_count":          len(words),
            "sentence_count":      len(sentences),
            "avg_word_length":     sum(len(w) for w in words) / len(words) if words else 0,
            "avg_sentence_length": (
                sum(len(s.split()) for s in sentences) / len(sentences) if sentences else 0
            ),
            "vocabulary_diversity": len(set(words)) / len(words) if words else 0,
            "special_char_ratio":   len(re.findall(r"[^a-zA-Z0-9\s]", processed)) / len(processed),
        })
    return metrics


def display_quality_report(m: dict):
    print("\n📊 Text Quality Report")
    print("=" * 40)
    print(f"Original length  : {m['original_length']:,} chars")
    print(f"Processed length : {m['processed_length']:,} chars")
    print(f"Compression ratio: {m['compression_ratio']:.2%}")
    if m.get("word_count", 0):
        print(f"Words            : {m['word_count']:,}")
        print(f"Sentences        : {m['sentence_count']:,}")
        print(f"Avg word length  : {m['avg_word_length']:.1f} chars")
        print(f"Avg sentence len : {m['avg_sentence_length']:.1f} words")
        print(f"Vocab diversity  : {m['vocabulary_diversity']:.2%}")
        vd = m["vocabulary_diversity"]
        tag = "✅ High" if vd > 0.4 else ("⚠️ Moderate" if vd > 0.2 else "❌ Low")
        print(f"Vocab assessment : {tag}")

## 8 · Keyword Extraction

In [ ]:
class KeywordExtractor:
    """KeyBERT with TF-IDF fallback."""

    def __init__(self):
        self.kw_model = kw_model

    # FIX: use top_n (not top_k) — KeyBERT API parameter name
    def _keybert(self, text: str, n: int = 10) -> list[dict]:
        if len(text.strip()) < 50:
            return []
        kws = self.kw_model.extract_keywords(
            text,
            keyphrase_ngram_range=(1, 3),
            stop_words="english",
            top_n=n,                   # ← fixed from top_k
            use_mmr=True,
            diversity=0.5,
        )
        return [{"keyword": k, "confidence": round(s, 3), "length": len(k.split())}
                for k, s in kws]

    # FIX: min_df=1 (was 2 — crashes on single-document corpus)
    def _tfidf(self, text: str, n: int = 10) -> list[dict]:
        sentences = sent_tokenize(text) or text.split("\n")
        if len(sentences) < 2:
            return []
        vec = TfidfVectorizer(
            max_features=1000, stop_words="english",
            ngram_range=(1, 3), max_df=0.85, min_df=1,   # ← fixed min_df
        )
        mat  = vec.fit_transform(sentences)
        names   = vec.get_feature_names_out()
        scores  = mat.mean(axis=0).A1
        pairs   = sorted(zip(names, scores), key=lambda x: x[1], reverse=True)
        return [{"keyword": k, "confidence": round(s, 3), "length": len(k.split())}
                for k, s in pairs[:n]]

    def extract(self, text: str, method: str = "keybert", n: int = 10) -> list[dict]:
        print("🔄 Extracting keywords …")
        kws = self._keybert(text, n) if method == "keybert" else []
        if not kws:
            kws = self._tfidf(text, n)
        print(f"  ✅ {len(kws)} keywords")
        return kws


keyword_extractor = KeywordExtractor()

## 9 · Named Entity Recognition

In [ ]:
class NamedEntityExtractor:
    CATEGORIES = {
        "PERSON": "People", "ORG": "Organizations",
        "GPE": "Geopolitical entities", "LOC": "Locations",
        "DATE": "Dates", "TIME": "Times", "MONEY": "Monetary values",
        "PERCENT": "Percentages", "PRODUCT": "Products",
        "EVENT": "Events", "WORK_OF_ART": "Works of art",
        "LAW": "Laws", "LANGUAGE": "Languages",
    }

    def __init__(self):
        self.nlp = nlp

    def extract(self, text: str) -> dict:
        if not text or self.nlp is None:
            return {}
        print("🔄 Extracting named entities …")
        doc = self.nlp(text)
        buckets: dict = {}
        for ent in doc.ents:
            t = ent.text.strip()
            if len(t) < 2 or ent.label_ not in self.CATEGORIES:
                continue
            buckets.setdefault(ent.label_, {})
            buckets[ent.label_][t] = buckets[ent.label_].get(t, 0) + 1

        result: dict = {}
        for label, counts in buckets.items():
            top = sorted(counts, key=counts.get, reverse=True)[:10]
            result[label] = {
                "category_name": self.CATEGORIES[label],
                "entities": [{"text": t, "count": counts[t]} for t in top],
            }

        total = sum(len(v["entities"]) for v in result.values())
        print(f"  ✅ {total} entities in {len(result)} categories")
        return result

    def summary(self, entities: dict) -> dict:
        counts = {k: len(v["entities"]) for k, v in entities.items()}
        all_e  = [(e["text"], e["count"], k)
                  for k, v in entities.items()
                  for e in v["entities"]]
        return {
            "total_categories": len(entities),
            "total_entities":   sum(counts.values()),
            "most_common_type": max(counts, key=counts.get) if counts else None,
            "most_common_entity": max(all_e, key=lambda x: x[1]) if all_e else None,
        }


entity_extractor = NamedEntityExtractor()

## 10 · Text Summarisation

In [ ]:
class TextSummarizer:
    MAX_CHUNK = 1024

    def __init__(self):
        self.model = summarizer

    def _chunk(self, text: str, max_len: int = MAX_CHUNK) -> list[str]:
        chunks, buf, buf_len = [], [], 0
        for s in sent_tokenize(text):
            if buf_len + len(s) > max_len and buf:
                chunks.append(" ".join(buf))
                buf, buf_len = [s], len(s)
            else:
                buf.append(s); buf_len += len(s)
        if buf:
            chunks.append(" ".join(buf))
        return chunks

    def summarise(self, text: str, style: str = "balanced") -> str:
        if not text or self.model is None:
            return ""
        print("🔄 Generating summary …")
        lims = {"brief": (30, 100), "detailed": (100, 250)}.get(style, (50, 150))
        summaries = []
        for chunk in self._chunk(text)[:3]:
            try:
                out = self.model(chunk, min_length=lims[0], max_length=lims[1],
                                  do_sample=False, num_beams=4, early_stopping=True)
                summaries.append(out[0]["summary_text"])
            except Exception as e:
                print(f"  ⚠️ chunk failed: {e}")
        final = " ".join(summaries)
        print(f"  ✅ {len(final)} chars")
        return final

    def key_sentences(self, text: str, n: int = 3) -> list[str]:
        sentences = sent_tokenize(text)
        if len(sentences) <= n:
            return sentences
        scored = []
        for i, s in enumerate(sentences):
            sc  = min(len(s.split()), 20) / 20 * 0.3
            sc += 0.4 if i < len(sentences) * 0.3 else (0.3 if i > len(sentences) * 0.7 else 0)
            scored.append((s, sc))
        scored.sort(key=lambda x: x[1], reverse=True)
        return [s for s, _ in scored[:n]]


text_summarizer = TextSummarizer()

## 11 · Language Detection & Complexity

In [ ]:
def detect_language(text: str) -> str:
    if not text or len(text.strip()) < 50 or nlp is None:
        return "unknown"
    doc = nlp(text[:1000])
    en_words = {"the","and","is","in","to","of","a","that","it","with"}
    tokens = [t.text.lower() for t in doc if t.is_alpha]
    if tokens:
        ratio = sum(1 for w in tokens if w in en_words) / len(tokens)
        if ratio > 0.1:
            return "en"
    return "unknown"


def analyse_complexity(text: str) -> dict:
    if not text:
        return {}
    words     = text.split()
    sentences = sent_tokenize(text)
    unique    = set(w.lower() for w in words if w.isalpha())
    avg_wps   = len(words) / len(sentences) if sentences else 0
    complexity = "high" if avg_wps > 20 else ("medium" if avg_wps > 15 else "low")
    return {
        "avg_words_per_sentence": round(avg_wps, 2),
        "avg_chars_per_word":     round(sum(len(w) for w in words) / len(words), 2) if words else 0,
        "vocabulary_richness":    round(len(unique) / len(words), 3) if words else 0,
        "complexity_level":       complexity,
        "total_words":            len(words),
        "total_sentences":        len(sentences),
        "unique_words":           len(unique),
    }

## 12 · Metadata Generator

In [ ]:
class MetadataGenerator:

    def _generate_title(self, text: str, keywords: list[dict], filename: str) -> str:
        for s in sent_tokenize(text)[:3]:
            s = s.strip()
            if len(s.split()) <= 15 and not s.endswith(".") and s[:1].isupper():
                return s
        if keywords:
            candidate = " ".join(k["keyword"] for k in keywords[:3]).title()
            if len(candidate) <= 100:
                return candidate
        return Path(filename).stem.replace("_", " ").replace("-", " ").title()

    def _quality_scores(self, text_len: int, raw_meta: dict,
                        n_keywords: int, n_entities: int) -> dict:
        ext = min(0.4*(text_len>1000) + 0.3*(text_len>0) + 0.2*bool(raw_meta.get("title")), 1.0)
        tq  = min(0.4*(text_len>500)  + 0.3*(text_len>100) + 0.3, 1.0)
        comp = sum([n_keywords>0, n_entities>0, text_len>200, text_len//5>50]) / 4
        return {
            "extraction_confidence": round(ext,  2),
            "text_quality_score":    round(tq,   2),
            "completeness_score":    round(comp, 2),
        }

    def _structure_entities(self, entities: dict) -> dict:
        mapping = {
            "PERSON": "people", "ORG": "organizations",
            "GPE": "locations", "LOC": "locations",
            "DATE": "dates",    "TIME": "dates",
        }
        out = {"people": [], "organizations": [], "locations": [], "dates": [], "other": {}}
        for label, data in entities.items():
            names = [e["text"] for e in data["entities"]]
            cat = mapping.get(label)
            if cat:
                out[cat].extend(names)
            else:
                out["other"][label] = names
        return out

    def generate(self, text: str, raw_meta: dict, filename: str) -> dict:
        if not text:
            return self._empty(filename)
        print("\n🔬 Generating metadata …")

        keywords  = keyword_extractor.extract(text, n=15)
        entities  = entity_extractor.extract(text)
        summary   = text_summarizer.summarise(text)
        key_sents = text_summarizer.key_sentences(text)
        language  = detect_language(text)
        complexity= analyse_complexity(text)
        title     = self._generate_title(text, keywords, filename)
        n_ents    = sum(len(v["entities"]) for v in entities.values())
        quality   = self._quality_scores(len(text), raw_meta, len(keywords), n_ents)

        meta = {
            "document_info": {
                "title":           title,
                "filename":        filename,
                "file_type":       raw_meta.get("file_extension", Path(filename).suffix),
                "author":          raw_meta.get("author", ""),
                "page_count":      raw_meta.get("page_count", ""),
                "creation_date":   str(raw_meta.get("creation_time", "")),
                "processing_date": pd.Timestamp.now().isoformat(),
            },
            "content_analysis": {
                "language":         language,
                "word_count":       complexity.get("total_words", 0),
                "character_count":  len(text),
                "sentence_count":   complexity.get("total_sentences", 0),
                "complexity_metrics": complexity,
            },
            "semantic_metadata": {
                "keywords":        [k["keyword"] for k in keywords],
                "keyword_details": keywords,
                "summary":         summary,
                "key_sentences":   key_sents,
            },
            "entities":        self._structure_entities(entities),
            "quality_metrics": quality,
            "raw_extraction_metadata": raw_meta,
        }

        print(f"  ✅ {len(keywords)} keywords  |  {n_ents} entities  |"
              f"  quality {quality['completeness_score']:.0%}")
        return meta

    def _empty(self, filename: str) -> dict:
        return {
            "document_info": {"title": Path(filename).stem, "filename": filename,
                               "processing_date": pd.Timestamp.now().isoformat()},
            "content_analysis": {"language": "unknown", "word_count": 0},
            "semantic_metadata": {"keywords": [], "summary": "", "key_sentences": []},
            "entities": {"people": [], "organizations": [], "locations": [], "dates": [], "other": {}},
            "quality_metrics": {"extraction_confidence": 0, "text_quality_score": 0, "completeness_score": 0},
        }


metadata_generator = MetadataGenerator()

## 13 · Metadata Export

In [ ]:
class MetadataExporter:

    def to_json(self, meta: dict, pretty: bool = True) -> str:
        return json.dumps(meta, indent=2 if pretty else None,
                          ensure_ascii=False, default=str)

    def to_flat_dict(self, meta: dict) -> dict:
        flat: dict = {}
        def _flatten(d, prefix=""):
            for k, v in d.items():
                key = f"{prefix}{k}" if prefix else k
                if isinstance(v, dict):
                    _flatten(v, f"{key}_")
                elif isinstance(v, list):
                    flat[key] = "; ".join(v) if v and isinstance(v[0], str) else str(v)
                else:
                    flat[key] = str(v) if v is not None else ""
        _flatten(meta)
        return flat

    def save(self, meta: dict, base_name: str, fmt: str = "json"):
        if fmt == "json":
            path = f"{base_name}.json"
            with open(path, "w", encoding="utf-8") as f:
                f.write(self.to_json(meta))
        elif fmt == "csv":
            path = f"{base_name}.csv"
            pd.DataFrame([self.to_flat_dict(meta)]).to_csv(path, index=False)
        else:
            print(f"❌ Unknown format: {fmt}")
            return
        print(f"✅ Saved → {path}")


metadata_exporter = MetadataExporter()

## 14 · End-to-End Pipeline

In [ ]:
def process_document(file_path: str,
                     export_fmt: str = "json",
                     save: bool = True) -> dict | None:
    """
    Full pipeline: load → extract → preprocess → analyse → export.

    Parameters
    ----------
    file_path  : path to PDF / DOCX / TXT / PNG / JPG
    export_fmt : 'json' or 'csv'
    save       : write output file to disk

    Returns
    -------
    metadata dict, or None on failure
    """
    print("=" * 60)
    print(f"🚀  {Path(file_path).name}")
    print("=" * 60)

    file_info = load_file_info(file_path)
    if not file_info:
        return None

    # 1 — Extract
    print("\n📋 STEP 1 · Text Extraction")
    text, raw_meta = extract_text_from_file(file_info)
    if not text.strip():
        print("❌ No text could be extracted.")
        return None

    # 2 — Preprocess
    print("\n🧹 STEP 2 · Preprocessing")
    pp = text_preprocessor.preprocess(text)
    clean = pp["processed_text"]

    # 3 — Quality
    print("\n📊 STEP 3 · Quality Assessment")
    quality_m = assess_text_quality(text, clean)
    display_quality_report(quality_m)

    # 4 — Metadata
    print("\n🔬 STEP 4 · Semantic Analysis")
    metadata = metadata_generator.generate(clean, raw_meta, file_info["name"])

    # 5 — Export
    if save:
        print("\n💾 STEP 5 · Exporting")
        base = f"metadata_{Path(file_info['name']).stem}"
        metadata_exporter.save(metadata, base, export_fmt)

    q = metadata["quality_metrics"]
    print(f"\n✅ Done  |  words: {metadata['content_analysis']['word_count']:,}"
          f"  |  quality: {q['completeness_score']:.0%}")
    return metadata


def process_batch(file_paths: list[str], export_fmt: str = "json") -> dict:
    """Process multiple documents and return all results."""
    results: dict = {}
    for i, path in enumerate(file_paths, 1):
        print(f"\n[{i}/{len(file_paths)}] {Path(path).name}")
        try:
            m = process_document(path, export_fmt)
            if m:
                results[Path(path).name] = m
        except Exception as e:
            print(f"  ❌ {e}")
    print(f"\n🎉 Batch done: {len(results)}/{len(file_paths)} succeeded")
    return results

## 15 · Display & Visualisation

In [ ]:
def display_metadata_summary(meta: dict):
    """Print a formatted human-readable summary."""    if not meta:
        print("❌ No metadata."); return

    print(f"\n{'🔍 METADATA REPORT':^60}")
    print("=" * 60)

    di = meta["document_info"]
    print(f"\n📄 Document Info")
    print(f"  Title    : {di.get('title','')}")
    print(f"  File     : {di.get('filename','')}")
    print(f"  Author   : {di.get('author','—')}")
    print(f"  Pages    : {di.get('page_count','—')}")
    print(f"  Processed: {str(di.get('processing_date',''))[:19]}")

    ca = meta["content_analysis"]
    print(f"\n📊 Content")
    print(f"  Language   : {ca.get('language','').upper()}")
    print(f"  Words      : {ca.get('word_count',0):,}")
    print(f"  Sentences  : {ca.get('sentence_count',0):,}")
    cm = ca.get("complexity_metrics", {})
    print(f"  Complexity : {cm.get('complexity_level','—').title()}")
    print(f"  Vocab rich : {cm.get('vocabulary_richness',0):.2%}")

    sm = meta["semantic_metadata"]
    print(f"\n🏷️  Keywords ({len(sm['keywords'])})")
    for i, kw in enumerate(sm["keywords"][:10], 1):
        print(f"  {i:2d}. {kw}")

    ents = meta["entities"]
    print(f"\n👥 Entities")
    for cat in ("people", "organizations", "locations", "dates"):
        items = ents.get(cat, [])
        if items:
            print(f"  {cat.title():<15}: {', '.join(items[:5])}")
            if len(items) > 5: print(f"  {'':15}  … +{len(items)-5} more")

    if sm.get("summary"):
        print(f"\n📝 Summary")
        print(textwrap.fill(sm["summary"], width=70, initial_indent="  ",
                             subsequent_indent="  "))

    q = meta["quality_metrics"]
    print(f"\n⭐ Quality")
    print(f"  Extraction confidence : {q['extraction_confidence']:.0%}")
    print(f"  Text quality          : {q['text_quality_score']:.0%}")
    print(f"  Completeness          : {q['completeness_score']:.0%}")
    overall = sum(q.values()) / 3
    tag = "🌟 Excellent" if overall>=0.8 else ("✅ Good" if overall>=0.6 else
          ("⚠️ Fair" if overall>=0.4 else "❌ Poor"))
    print(f"\n  Overall: {tag} ({overall:.0%})")


def visualise_metadata(meta: dict):
    """Four-panel dashboard visualisation."""    if not meta:
        return

    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("Document Metadata Dashboard", fontsize=15, fontweight="bold")
    ax1, ax2, ax3, ax4 = axes.flat

    # 1 — Keyword confidence
    kd = meta["semantic_metadata"].get("keyword_details", [])[:10]
    if kd:
        kws   = [k["keyword"]    for k in kd]
        confs = [k["confidence"] for k in kd]
        ax1.barh(range(len(kws)), confs, color=sns.color_palette("husl", len(kws)))
        ax1.set_yticks(range(len(kws))); ax1.set_yticklabels(kws, fontsize=8)
        ax1.set_xlabel("Confidence"); ax1.set_title("Top Keywords")
        ax1.invert_yaxis()

    # 2 — Entity distribution
    ents = meta["entities"]
    ent_counts = {
        "People":        len(ents.get("people", [])),
        "Organisations": len(ents.get("organizations", [])),
        "Locations":     len(ents.get("locations", [])),
        "Dates":         len(ents.get("dates", [])),
        "Other":         sum(len(v) for v in ents.get("other", {}).values()),
    }
    ent_counts = {k: v for k, v in ent_counts.items() if v}
    if ent_counts:
        ax2.pie(ent_counts.values(), labels=ent_counts.keys(), autopct="%1.0f%%",
                colors=sns.color_palette("husl", len(ent_counts)))
        ax2.set_title("Entity Distribution")

    # 3 — Quality metrics
    q = meta["quality_metrics"]
    labels = ["Extraction\nConf.", "Text\nQuality", "Completeness"]
    vals   = [q["extraction_confidence"], q["text_quality_score"], q["completeness_score"]]
    bars   = ax3.bar(labels, vals, color=["steelblue", "seagreen", "tomato"])
    ax3.set_ylim(0, 1); ax3.set_ylabel("Score"); ax3.set_title("Quality Metrics")
    for b, v in zip(bars, vals):
        ax3.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.2f}",
                 ha="center", fontsize=9)

    # 4 — Content stats
    ca    = meta["content_analysis"]
    slabs = ["Words", "Sentences", "Chars / 100"]
    svals = [ca.get("word_count",0), ca.get("sentence_count",0),
             ca.get("character_count",0)//100]
    ax4.bar(slabs, svals, color=["gold", "orange", "coral"])
    ax4.set_ylabel("Count"); ax4.set_title("Content Stats")
    for i, v in enumerate(svals):
        ax4.text(i, v + max(svals)*0.01, f"{v:,}", ha="center", fontsize=9)

    plt.tight_layout()
    plt.show()

## 16 · Quick Demo (built-in sample text)

In [ ]:
SAMPLE_TEXT = """
Artificial Intelligence and Machine Learning in Healthcare: A Comprehensive Review

The integration of artificial intelligence (AI) and machine learning (ML) technologies in healthcare
has revolutionised medical diagnosis, treatment planning, and patient care. This comprehensive review
examines current applications, challenges, and future prospects of AI-driven healthcare solutions.

Dr. Sarah Johnson from Stanford University Medical Center and Prof. Michael Chen from MIT have
conducted extensive research on AI applications in radiology and pathology. Their findings,
published in the Journal of Medical Informatics in 2023, demonstrate significant improvements
in diagnostic accuracy.

Key applications include medical imaging analysis, drug discovery, personalised treatment
recommendations, clinical decision support systems, and predictive analytics for patient outcomes.

The technology has shown particular promise in detecting early-stage cancers, with accuracy rates
exceeding 95 percent in clinical trials conducted across hospitals in Boston, New York, and
San Francisco between January 2022 and December 2023.

Challenges remain in data privacy, algorithmic bias, regulatory compliance, and the need for
extensive clinical validation. Healthcare organisations must address these concerns while
implementing AI solutions to ensure patient safety and maintain public trust.
""".strip()

# Save to temp file and run the full pipeline
import tempfile, os
with tempfile.NamedTemporaryFile(mode="w", suffix=".txt",
                                  delete=False, encoding="utf-8") as tmp:
    tmp.write(SAMPLE_TEXT)
    tmp_path = tmp.name

sample_metadata = process_document(tmp_path, save=False)
os.remove(tmp_path)

if sample_metadata:
    display_metadata_summary(sample_metadata)
    visualise_metadata(sample_metadata)

## 17 · Process Your Own File

In [ ]:
# ─────────────────────────────────────────────────────────────
# Change this path to your document and run the cell.
# Supported: .pdf  .docx  .txt  .png  .jpg  .jpeg
# ─────────────────────────────────────────────────────────────

FILE_PATH = "/content/your_document.pdf"   # ← edit this line

metadata = process_document(FILE_PATH, export_fmt="json", save=True)

if metadata:
    display_metadata_summary(metadata)
    visualise_metadata(metadata)

    # Preview the JSON export
    print("\n📦 JSON preview (first 600 chars):")
    print(metadata_exporter.to_json(metadata)[:600], "…")